# MYTax QLoRA Fine-tuning on Google Colab

Fine-tune Gemma 3 12B on Malaysia tax Q&A data using QLoRA (4-bit).

**Requirements:** Colab T4 GPU (free tier, 15GB VRAM)

**Steps:**
1. Upload `merged.jsonl` to Colab
2. Run all cells
3. Download the GGUF file
4. Import into Ollama locally

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth transformers datasets trl peft bitsandbytes

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Step 2: Upload training data
from google.colab import files
uploaded = files.upload()  # Upload merged.jsonl here
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH}")

In [ ]:
# Step 3: Load and prepare dataset
import json

SYSTEM_PROMPT = """You are MYTax AI, a Malaysia tax expert assistant for YA2025.
You answer questions about Malaysian taxation accurately and concisely.
You can respond in English, Chinese, or Malay. Add disclaimers when appropriate."""

examples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        msgs = obj.get("messages", [])
        if len(msgs) >= 2:
            examples.append({
                "conversations": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": msgs[0]["content"]},
                    {"role": "assistant", "content": msgs[1]["content"]},
                ]
            })

print(f"Loaded {len(examples)} training examples")
print(f"Sample Q: {examples[0]['conversations'][1]['content'][:100]}")

In [ ]:
# Step 4: Load model with QLoRA
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="google/gemma-3-12b-it",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model.print_trainable_parameters()

In [ ]:
# Step 5: Prepare dataset for training
from datasets import Dataset

dataset = Dataset.from_list(examples)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, remove_columns=["conversations"])
print(f"Dataset: {len(dataset)} examples")
print(f"Sample: {dataset[0]['text'][:300]}...")

In [ ]:
# Step 6: Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=3,
        warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=2048,
)

trainer.train()

In [ ]:
# Step 7: Export to GGUF
model.save_pretrained_gguf(
    "./output",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF export complete!")

# Create Modelfile for Ollama
import glob
gguf_files = glob.glob("./output/*.gguf")
if gguf_files:
    gguf_name = gguf_files[0].split("/")[-1]
    with open("./output/Modelfile", "w") as f:
        f.write(f'FROM ./{gguf_name}\n\n')
        f.write(f'SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"\n\n')
        f.write('PARAMETER temperature 0.7\n')
        f.write('PARAMETER top_p 0.9\n')
        f.write('PARAMETER num_ctx 4096\n')
    print(f"Modelfile created")

In [ ]:
# Step 8: Download the GGUF file
import glob
from google.colab import files

gguf_files = glob.glob("./output/*.gguf")
for f in gguf_files:
    print(f"Downloading: {f}")
    files.download(f)

# Also download Modelfile
files.download("./output/Modelfile")

print("\n" + "="*60)
print("DONE! Next steps on your local machine:")
print("  1. Place .gguf and Modelfile in the same folder")
print("  2. cd to that folder")
print("  3. ollama create mytax-gemma4-finetuned -f Modelfile")
print("  4. ollama run mytax-gemma4-finetuned")
print("  5. Update .env.local: OLLAMA_MODEL=mytax-gemma4-finetuned")
print("="*60)